In [1]:
import os
import numpy
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, text
import datetime
import pymongo
import pprint
import json
import certifi

print(f"Running SQL Alchemy Version: {sqlalchemy.__version__}")
print(f"Running PyMongo Version: {pymongo.__version__}")

Running SQL Alchemy Version: 2.0.34
Running PyMongo Version: 4.16.0


#### Declare & Assign Connection Variables for the MySQL Server & Databases with which You'll be Working 

In [13]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "P0ssward"

src_dbname = "adventureworks"
dst_dbname = "adventureworks_dw"

mysql_args = {
    "uid" : "root",
    "pwd" : "P0ssward",
    "hostname" : "localhost",  #"wna8fw-mysql.mysql.database.azure.com",
    "dbname" : "adventureworks_dw"
}

# The 'cluster_location' must either be "atlas" or "local".
mongodb_args = {
    "user_name" : "jtupitza",
    "password" : "Passw0rd1234",
    "cluster_name" : "sandbox",
    "cluster_subnet" : "zibbf",
    "cluster_location" : "atlas", # "local"
    "db_name" : "adventureworks_project1"
}

#### Define Functions for Getting Data From and Setting Data Into Databases

In [3]:
def get_dataframe_m(user_id, pwd, host_name, db_name, sql_query):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(sql_query, connection);
    connection.close()
    
    return dframe

def set_dataframe_m(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()
    


In [4]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    
def set_dataframe(df, table_name, pk_column, db_operation, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

#### Create the New Data Warehouse database

In [5]:
conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE {dst_dbname};"))

connection.close()

#### Create & Populate Dimension Tables from SQL

In [6]:
sql_employees = "SELECT * FROM adventureworks.dim_employee_vw;"
df_employees = get_dataframe_m(user_id, pwd, host_name, src_dbname, sql_employees)
df_employees.head(2)

,EmployeeID,NationalIDNumber,LoginID,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,EmailPromotion,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,14417807,adventure-works\guy1,16.0,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,0,320-555-0195,1972-05-15,M,M,1996-07-31,b'\x00',21,30,b'\x01'
1,2,253022876,adventure-works\kevin0,6.0,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,2,150-555-0189,1977-06-03,S,M,1997-02-26,b'\x00',42,41,b'\x01'


In [23]:
sql_vendors = "SELECT * FROM adventureworks.dim_vendors_vw;"
df_vendors = get_dataframe_m(user_id, pwd, host_name, src_dbname, sql_vendors)
df_vendors.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,PostalCode
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',Main Office,683 Larch Ct.,None,Salt Lake City,UT,Utah,84101
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',Main Office,8547 Catherine Way,None,Tacoma,WA,Washington,98403


In [17]:
sql_fact_po = "SELECT * FROM adventureworks.fact_purchase_orders_vw;"
df_fact_po = get_dataframe_m(user_id, pwd, host_name, src_dbname, sql_fact_po)
df_fact_po.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ProductID,OrderQty,UnitPrice,LineTotal,OrderDate,...,ShipRate,ShipDate,SubTotal,TaxAmt,Freight,TotalDue,DueDate,ReceivedQty,RejectedQty,StockedQty
0,1,0,4,244,83,1,4,50.2600,201.0400,2001-05-17,...,2.99,2001-05-26,201.0400,16.0832,5.0260,222.1492,2001-05-31,3.0,0.0,3.0
1,2,0,1,231,32,360,3,45.5805,136.7415,2001-05-17,...,1.49,2001-05-26,272.1015,21.7681,6.8025,300.6721,2001-05-31,3.0,0.0,3.0


#### Create and Populate Dimension Tables from CSV file

In [15]:
vendor_csv = 'adventureworks_dim_customers.csv'
df_customers = pd.read_csv(vendor_csv)
df_customers.head(2)

,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,IsOnlyStateProvinceFlag,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,2251 Elliot Avenue,NaN,Seattle,WA,Washington,0,98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,7943 Walnut Ave,NaN,Renton,WA,Washington,0,98055,US,United States,North America,Northwest


#### Create and Populate Dimension Table from Source MongoDB Collection

In [22]:
client = get_mongo_client(**mongodb_args)

query = {} # Select all elements (columns), and all documents (rows).
collection = "products"

df_products = get_mongo_dataframe(client, mongodb_args["db_name"], collection, query)
df_products.head(2)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,DaysToManufacture,ProductLine,Class,Style,ProductCategory,ProductSubcategory,ProductModel,SellStartDate,SellEndDate,DiscontinuedDate
0,1,Adjustable Race,AR-5381, , ,None,1000,750,0.0,0.0,...,0,None,None,None,None,None,None,1998-06-01T00:00:00.000,None,None
1,2,Bearing Ball,BA-8327, , ,None,1000,750,0.0,0.0,...,0,None,None,None,None,None,None,1998-06-01T00:00:00.000,None,None


#### Get Data from Data Dimension Table

In [14]:
sql_dim_date = "SELECT date_key, full_date FROM adventureworks.dim_date;"
df_dim_date = get_sql_dataframe(sql_dim_date, **mysql_args)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
df_dim_date.head(2)

,date_key,full_date
0,20000101,2000-01-01
1,20000102,2000-01-02


Transormations

In [19]:
#customers
# 1. Create a List that enumerates the names of each column you wish to remove (drop) from the Pandas DataFrame
drop_cols = ['AddressLine1','AddressLine2']
df_customers.drop(drop_cols, axis=1, inplace=True)

df_customers.head(2)

,CustomerID,AccountNumber,CustomerType,AddressType,City,StateProvinceCode,State_Province,IsOnlyStateProvinceFlag,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,Seattle,WA,Washington,0,98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,Renton,WA,Washington,0,98055,US,United States,North America,Northwest


In [20]:
#employees
drop_cols = ['LoginID', 'EmailPromotion']
df_employees.drop(drop_cols, axis=1, inplace=True)

df_employees.head(2)

,EmployeeID,NationalIDNumber,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,14417807,16.0,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15,M,M,1996-07-31,b'\x00',21,30,b'\x01'
1,2,253022876,6.0,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03,S,M,1997-02-26,b'\x00',42,41,b'\x01'


In [24]:
#products
drop_cols = ['SizeUnitMeasureCode', 'WeightUnitMeasureCode', 'Weight', 'ProductLine', 'Style', 'ProductCategory', 'ProductSubcategory', 'ProductModel', 'DiscontinuedDate']
df_products.drop(drop_cols, axis=1, inplace=True)

df_products.head(2)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,DaysToManufacture,Class,SellStartDate,SellEndDate
0,1,Adjustable Race,AR-5381, , ,None,1000,750,0.0,0.0,None,0,None,1998-06-01T00:00:00.000,None
1,2,Bearing Ball,BA-8327, , ,None,1000,750,0.0,0.0,None,0,None,1998-06-01T00:00:00.000,None


In [25]:
#vendors
drop_cols = ['StateProvinceCode']
df_vendors.drop(drop_cols, axis=1, inplace=True)

df_vendors.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,AddressType,AddressLine1,AddressLine2,City,State_Province,PostalCode
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',Main Office,683 Larch Ct.,None,Salt Lake City,Utah,84101
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',Main Office,8547 Catherine Way,None,Tacoma,Washington,98403


Lookup the Surrogate Primary Key (date_key)

In [49]:
#rename date columns 
df_fact_po = df_fact_po.rename(columns = {'OrderDate':'order_date','ShipDate':'ship_date','DueDate':'due_date'})

In [50]:
df_dim_order_date = df_dim_date.rename(columns={"date_key" : "order_date_key", "full_date" : "order_date"})
df_fact_po.order_date = df_fact_po.order_date.astype('datetime64[ns]').dt.date
df_fact_po = pd.merge(df_fact_po, df_dim_order_date, on='order_date', how='left')
df_fact_po.drop(['order_date'], axis=1, inplace=True)
df_fact_po.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ProductID,OrderQty,UnitPrice,LineTotal,ShipMethod,...,ship_date,SubTotal,TaxAmt,Freight,TotalDue,due_date,ReceivedQty,RejectedQty,StockedQty,order_date_key
0,1,0,4,244,83,1,4,50.2600,201.0400,OVERSEAS - DELUXE,...,2001-05-26,201.0400,16.0832,5.0260,222.1492,2001-05-31,3.0,0.0,3.0,20010517
1,2,0,1,231,32,360,3,45.5805,136.7415,CARGO TRANSPORT 5,...,2001-05-26,272.1015,21.7681,6.8025,300.6721,2001-05-31,3.0,0.0,3.0,20010517


In [51]:
df_dim_ship_date = df_dim_date.rename(columns={"date_key" : "ship_date_key", "full_date" : "ship_date"})
df_fact_po.ship_date = df_fact_po.ship_date.astype('datetime64[ns]').dt.date
df_fact_po = pd.merge(df_fact_po, df_dim_ship_date, on='ship_date', how='left')
df_fact_po.drop(['ship_date'], axis=1, inplace=True)
df_fact_po.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ProductID,OrderQty,UnitPrice,LineTotal,ShipMethod,...,SubTotal,TaxAmt,Freight,TotalDue,due_date,ReceivedQty,RejectedQty,StockedQty,order_date_key,ship_date_key
0,1,0,4,244,83,1,4,50.2600,201.0400,OVERSEAS - DELUXE,...,201.0400,16.0832,5.0260,222.1492,2001-05-31,3.0,0.0,3.0,20010517,20010526
1,2,0,1,231,32,360,3,45.5805,136.7415,CARGO TRANSPORT 5,...,272.1015,21.7681,6.8025,300.6721,2001-05-31,3.0,0.0,3.0,20010517,20010526


In [52]:
df_dim_due_date = df_dim_date.rename(columns={"date_key" : "due_date_key", "full_date" : "due_date"})
df_fact_po.due_date = df_fact_po.due_date.astype('datetime64[ns]').dt.date
df_fact_po = pd.merge(df_fact_po, df_dim_due_date, on='due_date', how='left')
df_fact_po.drop(['due_date'], axis=1, inplace=True)
df_fact_po.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ProductID,OrderQty,UnitPrice,LineTotal,ShipMethod,...,SubTotal,TaxAmt,Freight,TotalDue,ReceivedQty,RejectedQty,StockedQty,order_date_key,ship_date_key,due_date_key
0,1,0,4,244,83,1,4,50.2600,201.0400,OVERSEAS - DELUXE,...,201.0400,16.0832,5.0260,222.1492,3.0,0.0,3.0,20010517,20010526,20010531
1,2,0,1,231,32,360,3,45.5805,136.7415,CARGO TRANSPORT 5,...,272.1015,21.7681,6.8025,300.6721,3.0,0.0,3.0,20010517,20010526,20010531


Load the Transformed DataFrames into the New Data Warehouse by Creating New Tables¶

In [57]:
dataframe = df_customers
table_name = 'dim_customers'
primary_key = 'customer_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [54]:
dataframe = df_employees
table_name = 'dim_employees'
primary_key = 'employee_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [55]:
dataframe = df_products
table_name = 'dim_products'
primary_key = 'product_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [56]:
dataframe = df_vendors
table_name = 'dim_vendors'
primary_key = 'vendor_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)

In [ ]:
dataframe = df_fact_po
table_name = 'fact_purchase_orders'
primary_key = 'fact_purchase_orders_key'
db_operation = "insert"

set_dataframe(dataframe, table_name, primary_key, db_operation, **mysql_args)